In [1]:
import json
import numpy as np
import datasets

with open("marco_qrels.json") as f:
    qrels = json.load(f)
    
ds = datasets.load_from_disk("msmarco_raw")

/home/zhichaog/anaconda3/envs/neural-sparse-mdbert/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
q_ids = ds["q_id"]
d_ids = ds["d_ids"]

In [20]:
with open("q_ids.json", "w") as f:
    json.dump(q_ids, f)
    
with open("d_ids.json", "w") as f:
    json.dump(d_ids, f)

In [34]:
# load scores


score_bge_gemma2 = np.load("scores/msmarco_raw/bge_gemma2_reranker_scores.npy")
score_minilm = np.load("scores/msmarco_raw/minilm_scores.npy").astype(np.float64)
score_3 = np.load("scores/msmarco_raw/llm-3_scores.npy").astype(np.float64)
score_4 = np.load("scores/msmarco_raw/llm-4_scores.npy").astype(np.float64)
score_5 = np.load("scores/msmarco_raw/llm-5_scores.npy").astype(np.float64)
score_6 = np.load("scores/msmarco_raw/llm-6_scores.npy").astype(np.float64)

In [38]:
from tqdm import tqdm
from beir.retrieval.evaluation import EvaluateRetrieval

def eval_score(scores, return_run_res = False, sample_num=1000000):
    run_res = dict(zip(q_ids[:sample_num], map(dict, map(zip, d_ids[:sample_num], scores[:sample_num]))))
    if return_run_res:
        return run_res
    else:
        return EvaluateRetrieval.evaluate(qrels, run_res, [10])
    
print(eval_score(score_bge_gemma2, sample_num=2000))
print(eval_score(score_minilm, sample_num=2000))
print(eval_score(score_3, sample_num=2000))
print(eval_score(score_4, sample_num=2000))
print(eval_score(score_5, sample_num=2000))
print(eval_score(score_6, sample_num=2000))

({'NDCG@10': 0.48027}, {'MAP@10': 0.40239}, {'Recall@10': 0.719}, {'P@10': 0.07605})
({'NDCG@10': 0.44527}, {'MAP@10': 0.36964}, {'Recall@10': 0.67696}, {'P@10': 0.0715})
({'NDCG@10': 0.45896}, {'MAP@10': 0.38254}, {'Recall@10': 0.69358}, {'P@10': 0.07315})
({'NDCG@10': 0.37927}, {'MAP@10': 0.30288}, {'Recall@10': 0.61662}, {'P@10': 0.06525})
({'NDCG@10': 0.36684}, {'MAP@10': 0.29533}, {'Recall@10': 0.58796}, {'P@10': 0.06195})
({'NDCG@10': 0.40321}, {'MAP@10': 0.32832}, {'Recall@10': 0.63292}, {'P@10': 0.06675})


In [39]:
print(eval_score(score_bge_gemma2))

({'NDCG@10': 0.47726}, {'MAP@10': 0.40189}, {'Recall@10': 0.70974}, {'P@10': 0.07402})
